In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.optimize import minimize, minimize_scalar
from matplotlib import animation
from IPython.display import HTML

sns.set_style('whitegrid')
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 7.1 Gradient Descent

### Algorithm: Gradient Descent

**Iterative update**:

$$x_{t+1} = x_t - \alpha \nabla f(x_t)$$

where:
- α: learning rate (step size)
- ∇f(x_t): gradient at current point

**Intuition**: Move in the direction of steepest descent!

### Variants:

1. **Batch GD**: Use all data
2. **Stochastic GD (SGD)**: Use one sample at a time
3. **Mini-batch GD**: Use small batches
4. **Momentum**: Accumulate velocity
5. **Adam**: Adaptive learning rates

In [ ]:
# Gradient descent on 1D function

def f_1d(x):
    """Objective function: f(x) = (x-2)² + 1"""
    return (x - 2)**2 + 1

def grad_f_1d(x):
    """Gradient: f'(x) = 2(x-2)"""
    return 2 * (x - 2)

def gradient_descent_1d(x0, learning_rate=0.1, n_iterations=20):
    """
    Perform gradient descent.
    
    Returns:
        history: List of (x, f(x)) tuples at each iteration
    """
    x = x0
    history = [(x, f_1d(x))]
    
    for _ in range(n_iterations):
        grad = grad_f_1d(x)
        x = x - learning_rate * grad
        history.append((x, f_1d(x)))
    
    return history

# Run gradient descent
x0 = 5.0
history = gradient_descent_1d(x0, learning_rate=0.1, n_iterations=20)

# Extract trajectory
x_vals = np.array([h[0] for h in history])
f_vals = np.array([h[1] for h in history])

print("Gradient Descent (1D)")
print(f"\nObjective: f(x) = (x-2)² + 1")
print(f"Starting point: x₀ = {x0}")
print(f"Learning rate: α = 0.1")
print(f"\nIteration    x         f(x)")
print("-" * 35)
for i, (x, fx) in enumerate(history[:10]):
    print(f"{i:5d}     {x:8.4f}   {fx:8.4f}")
print("...")
print(f"{len(history)-1:5d}     {x_vals[-1]:8.4f}   {f_vals[-1]:8.4f}")

print(f"\nTrue minimum: x* = 2, f(x*) = 1")
print(f"Found: x ≈ {x_vals[-1]:.4f}, f(x) ≈ {f_vals[-1]:.4f}")

# Visualize
x_plot = np.linspace(-1, 6, 300)
f_plot = f_1d(x_plot)

plt.figure(figsize=(12, 5))

# Function and trajectory
plt.subplot(1, 2, 1)
plt.plot(x_plot, f_plot, 'b-', linewidth=2, label='f(x)')
plt.plot(x_vals, f_vals, 'ro-', markersize=6, linewidth=1.5, label='GD trajectory')
plt.plot(2, 1, 'g*', markersize=20, label='True minimum')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Gradient Descent Trajectory')
plt.legend()
plt.grid(True, alpha=0.3)

# Convergence
plt.subplot(1, 2, 2)
plt.semilogy(f_vals - 1, 'r.-', linewidth=2, markersize=8)
plt.xlabel('Iteration')
plt.ylabel('f(x) - f(x*)')
plt.title('Convergence (log scale)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Gradient descent on 2D function

def f_2d(x):
    """Rosenbrock function: f(x,y) = (1-x)² + 100(y-x²)²"""
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

def grad_f_2d(x):
    """Gradient of Rosenbrock"""
    dx = -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2)
    dy = 200 * (x[1] - x[0]**2)
    return np.array([dx, dy])

def gradient_descent_2d(x0, learning_rate=0.001, n_iterations=100):
    """Gradient descent in 2D."""
    x = x0.copy()
    history = [x.copy()]
    
    for _ in range(n_iterations):
        grad = grad_f_2d(x)
        x = x - learning_rate * grad
        history.append(x.copy())
    
    return np.array(history)

# Run
x0 = np.array([-1.0, 1.0])
history = gradient_descent_2d(x0, learning_rate=0.001, n_iterations=500)

print("Gradient Descent (2D)")
print(f"\nObjective: Rosenbrock function")
print(f"f(x,y) = (1-x)² + 100(y-x²)²")
print(f"\nStarting point: x₀ = {x0}")
print(f"Learning rate: α = 0.001")
print(f"Iterations: 500")

print(f"\nTrue minimum: x* = [1, 1], f(x*) = 0")
print(f"Found: x ≈ {history[-1]}, f(x) ≈ {f_2d(history[-1]):.6f}")

# Visualize
x_range = np.linspace(-1.5, 1.5, 200)
y_range = np.linspace(-0.5, 1.5, 200)
X, Y = np.meshgrid(x_range, y_range)
Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = f_2d(np.array([X[i, j], Y[i, j]]))

plt.figure(figsize=(14, 6))

# Contour plot
plt.subplot(1, 2, 1)
levels = np.logspace(-1, 3, 20)
plt.contour(X, Y, Z, levels=levels, cmap='viridis')
plt.plot(history[:, 0], history[:, 1], 'r.-', markersize=4, linewidth=1, alpha=0.7)
plt.plot(history[0, 0], history[0, 1], 'go', markersize=10, label='Start')
plt.plot(1, 1, 'r*', markersize=15, label='True minimum')
plt.xlabel('x')
plt.ylabel('y')
plt.title('GD on Rosenbrock Function')
plt.legend()
plt.grid(True, alpha=0.3)

# Convergence
plt.subplot(1, 2, 2)
f_history = [f_2d(x) for x in history]
plt.semilogy(f_history, 'b.-', linewidth=2, markersize=6)
plt.xlabel('Iteration')
plt.ylabel('f(x)')
plt.title('Convergence (log scale)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRosenbrock is a challenging function with a narrow valley!")

In [ ]:
# Effect of learning rate

def test_learning_rates():
    """Test different learning rates."""
    x0 = np.array([5.0])
    learning_rates = [0.01, 0.1, 0.5, 0.9]
    
    plt.figure(figsize=(14, 8))
    
    for idx, lr in enumerate(learning_rates, 1):
        # Run GD
        history = gradient_descent_1d(x0[0], learning_rate=lr, n_iterations=20)
        x_vals = np.array([h[0] for h in history])
        f_vals = np.array([h[1] for h in history])
        
        plt.subplot(2, 2, idx)
        x_plot = np.linspace(-1, 6, 300)
        f_plot = f_1d(x_plot)
        
        plt.plot(x_plot, f_plot, 'b-', linewidth=2, alpha=0.3)
        plt.plot(x_vals, f_vals, 'ro-', markersize=6, linewidth=1.5)
        plt.plot(2, 1, 'g*', markersize=15)
        plt.xlabel('x')
        plt.ylabel('f(x)')
        plt.title(f'Learning rate α = {lr}')
        plt.grid(True, alpha=0.3)
        
        if lr == 0.9:
            plt.ylim([0, 20])
    
    plt.tight_layout()
    plt.show()
    
    print("Learning rate selection:")
    print("  α too small → slow convergence")
    print("  α too large → oscillation or divergence!")
    print("  α just right → fast, stable convergence")

test_learning_rates()

In [ ]:
# Gradient descent with momentum

def gradient_descent_momentum(x0, learning_rate=0.01, momentum=0.9, n_iterations=100):
    """
    GD with momentum:
    v_{t+1} = β*v_t + α*∇f(x_t)
    x_{t+1} = x_t - v_{t+1}
    """
    x = x0.copy()
    v = np.zeros_like(x0)
    history = [x.copy()]
    
    for _ in range(n_iterations):
        grad = grad_f_2d(x)
        v = momentum * v + learning_rate * grad
        x = x - v
        history.append(x.copy())
    
    return np.array(history)

# Compare vanilla GD vs GD with momentum
x0 = np.array([-1.0, 1.0])
n_iter = 200

history_vanilla = gradient_descent_2d(x0, learning_rate=0.001, n_iterations=n_iter)
history_momentum = gradient_descent_momentum(x0, learning_rate=0.001, momentum=0.9, n_iterations=n_iter)

print("Gradient Descent with Momentum")
print(f"\nVanilla GD after {n_iter} iterations:")
print(f"  x = {history_vanilla[-1]}")
print(f"  f(x) = {f_2d(history_vanilla[-1]):.6f}")

print(f"\nGD with momentum (β=0.9) after {n_iter} iterations:")
print(f"  x = {history_momentum[-1]}")
print(f"  f(x) = {f_2d(history_momentum[-1]):.6f}")

# Visualize
plt.figure(figsize=(14, 6))

# Trajectories
plt.subplot(1, 2, 1)
levels = np.logspace(-1, 3, 20)
plt.contour(X, Y, Z, levels=levels, cmap='viridis', alpha=0.3)
plt.plot(history_vanilla[:, 0], history_vanilla[:, 1], 'b.-', 
         markersize=3, linewidth=1, alpha=0.6, label='Vanilla GD')
plt.plot(history_momentum[:, 0], history_momentum[:, 1], 'r.-', 
         markersize=3, linewidth=1, alpha=0.6, label='GD + Momentum')
plt.plot(1, 1, 'g*', markersize=15, label='Minimum')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Comparison of Trajectories')
plt.legend()
plt.grid(True, alpha=0.3)

# Convergence
plt.subplot(1, 2, 2)
f_vanilla = [f_2d(x) for x in history_vanilla]
f_momentum = [f_2d(x) for x in history_momentum]
plt.semilogy(f_vanilla, 'b-', linewidth=2, label='Vanilla GD')
plt.semilogy(f_momentum, 'r-', linewidth=2, label='GD + Momentum')
plt.xlabel('Iteration')
plt.ylabel('f(x)')
plt.title('Convergence Rate')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nMomentum accelerates convergence in narrow valleys!")

## 7.2 Constrained Optimization and Lagrange Multipliers

### Problem:

Minimize f(x) subject to constraints g(x) = 0

$$\min_x f(x) \quad \text{s.t.} \quad g(x) = 0$$

### Lagrangian:

$$\mathcal{L}(x, \lambda) = f(x) + \lambda^T g(x)$$

### Optimality Conditions (KKT):

At optimal (x*, λ*):

1. $\nabla_x \mathcal{L} = \nabla f(x^*) + \lambda^* \nabla g(x^*) = 0$
2. $g(x^*) = 0$ (feasibility)

**Intuition**: At optimum, gradient of f is parallel to gradient of g!

In [ ]:
# Lagrange multipliers example
# Minimize f(x,y) = x² + y² subject to x + y = 1

from scipy.optimize import minimize

def objective(xy):
    """f(x,y) = x² + y²"""
    return xy[0]**2 + xy[1]**2

def constraint(xy):
    """g(x,y) = x + y - 1 = 0"""
    return xy[0] + xy[1] - 1

# Analytical solution using Lagrange multipliers
# L = x² + y² + λ(x + y - 1)
# ∂L/∂x = 2x + λ = 0  →  x = -λ/2
# ∂L/∂y = 2y + λ = 0  →  y = -λ/2
# ∂L/∂λ = x + y - 1 = 0
#
# Substituting: -λ/2 - λ/2 - 1 = 0  →  λ = -1
# Therefore: x* = y* = 1/2

x_analytical = np.array([0.5, 0.5])
f_analytical = objective(x_analytical)

print("Constrained Optimization Example")
print("\nProblem:")
print("  minimize f(x,y) = x² + y²")
print("  subject to x + y = 1")

print("\nAnalytical solution (Lagrange multipliers):")
print(f"  x* = {x_analytical}")
print(f"  f(x*) = {f_analytical:.4f}")
print(f"  Constraint satisfied? x + y = {x_analytical.sum():.4f}")

# Numerical solution
x0 = np.array([0.0, 1.0])
constraints = {'type': 'eq', 'fun': constraint}
result = minimize(objective, x0, constraints=constraints)

print("\nNumerical solution (scipy.optimize):")
print(f"  x* = {result.x}")
print(f"  f(x*) = {result.fun:.4f}")
print(f"  Success? {result.success}")

# Visualize
x_range = np.linspace(-0.5, 1.5, 200)
y_range = np.linspace(-0.5, 1.5, 200)
X, Y = np.meshgrid(x_range, y_range)
Z = X**2 + Y**2

plt.figure(figsize=(10, 8))
contour = plt.contour(X, Y, Z, levels=20, cmap='viridis')
plt.clabel(contour, inline=True, fontsize=8)

# Constraint: x + y = 1  →  y = 1 - x
x_constraint = np.linspace(-0.5, 1.5, 100)
y_constraint = 1 - x_constraint
plt.plot(x_constraint, y_constraint, 'r-', linewidth=3, label='Constraint: x+y=1')

# Optimal point
plt.plot(x_analytical[0], x_analytical[1], 'g*', markersize=20, label='Optimal x*')

# Gradients at optimal point
grad_f = 2 * x_analytical  # ∇f = [2x, 2y]
grad_g = np.array([1, 1])  # ∇g = [1, 1]

# Normalize for visualization
grad_f_norm = grad_f / np.linalg.norm(grad_f) * 0.3
grad_g_norm = grad_g / np.linalg.norm(grad_g) * 0.3

plt.quiver(x_analytical[0], x_analytical[1], grad_f_norm[0], grad_f_norm[1],
          color='blue', width=0.008, scale=1, scale_units='xy', label='∇f')
plt.quiver(x_analytical[0], x_analytical[1], grad_g_norm[0], grad_g_norm[1],
          color='orange', width=0.008, scale=1, scale_units='xy', label='∇g')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Constrained Optimization\n∇f and ∇g are parallel at optimum!')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.xlim([-0.5, 1.5])
plt.ylim([-0.5, 1.5])
plt.show()

print("\nKey insight: At x*, gradients ∇f and ∇g are parallel!")
print("This is exactly what Lagrange multipliers enforce.")

## 7.3 Convex Optimization

### Definition 7.2 (Convex Function)

f is **convex** if for all x, y and λ ∈ [0,1]:

$$f(\lambda x + (1-\lambda)y) \leq \lambda f(x) + (1-\lambda)f(y)$$

**Geometric interpretation**: Function lies below any line connecting two points.

### Properties:

1. **Any local minimum is a global minimum**
2. **Gradient descent is guaranteed to converge**
3. **Efficiently solvable**

### Examples:

**Convex**:
- x²
- |x|
- exp(x)
- -log(x) for x > 0
- ||x||₂²

**Not convex**:
- sin(x)
- x⁴ - x²
- Neural networks (generally)

In [ ]:
# Visualize convex vs non-convex

x = np.linspace(-3, 3, 300)

# Convex functions
f_convex1 = x**2
f_convex2 = np.abs(x)
f_convex3 = np.exp(x)

# Non-convex functions
f_nonconvex1 = np.sin(x)
f_nonconvex2 = x**4 - x**2

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Convex
axes[0, 0].plot(x, f_convex1, 'b-', linewidth=2)
axes[0, 0].set_title('f(x) = x² (Convex)', fontsize=12)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(x, f_convex2, 'b-', linewidth=2)
axes[0, 1].set_title('f(x) = |x| (Convex)', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(x, f_convex3, 'b-', linewidth=2)
axes[0, 2].set_title('f(x) = exp(x) (Convex)', fontsize=12)
axes[0, 2].set_ylim([0, 20])
axes[0, 2].grid(True, alpha=0.3)

# Non-convex
axes[1, 0].plot(x, f_nonconvex1, 'r-', linewidth=2)
axes[1, 0].set_title('f(x) = sin(x) (Non-convex)', fontsize=12)
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(x, f_nonconvex2, 'r-', linewidth=2)
axes[1, 1].set_title('f(x) = x⁴ - x² (Non-convex)', fontsize=12)
axes[1, 1].grid(True, alpha=0.3)

# Demonstrate convexity definition
x_demo = np.linspace(-2, 2, 100)
f_demo = x_demo**2

# Two points
x1, x2 = -1.5, 1.5
f1, f2 = x1**2, x2**2

# Line connecting them
x_line = np.linspace(x1, x2, 50)
f_line = f1 + (f2 - f1) * (x_line - x1) / (x2 - x1)

axes[1, 2].plot(x_demo, f_demo, 'b-', linewidth=2, label='f(x) = x²')
axes[1, 2].plot(x_line, f_line, 'g--', linewidth=2, label='Linear interpolation')
axes[1, 2].plot([x1, x2], [f1, f2], 'ro', markersize=10)
axes[1, 2].fill_between(x_line, f_demo[np.searchsorted(x_demo, x_line)], f_line, 
                        alpha=0.3, color='yellow', label='f ≤ line')
axes[1, 2].set_title('Convexity Definition', fontsize=12)
axes[1, 2].legend(fontsize=9)
axes[1, 2].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')

plt.tight_layout()
plt.show()

print("Convex functions: curve lies BELOW any connecting line")
print("Non-convex: can have multiple local minima!")

In [ ]:
# Demonstrate: Convex → unique global minimum
#             Non-convex → multiple local minima

def convex_function(x):
    return x**2 + 2*x + 1

def nonconvex_function(x):
    return x**4 - 4*x**2 + 2*x

# Test gradient descent from multiple starting points
starting_points = [-3, -1, 0, 1, 3]

def grad_convex(x):
    return 2*x + 2

def grad_nonconvex(x):
    return 4*x**3 - 8*x + 2

def gd_1d_general(x0, grad_func, lr=0.01, n_iter=100):
    x = x0
    for _ in range(n_iter):
        x = x - lr * grad_func(x)
    return x

# Run GD
x_plot = np.linspace(-3, 3, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convex
axes[0].plot(x_plot, convex_function(x_plot), 'b-', linewidth=2)
results_convex = []
for x0 in starting_points:
    x_final = gd_1d_general(x0, grad_convex, lr=0.1, n_iter=50)
    results_convex.append(x_final)
    axes[0].plot(x0, convex_function(x0), 'ro', markersize=8)
    axes[0].plot(x_final, convex_function(x_final), 'g*', markersize=15)
    axes[0].annotate('', xy=(x_final, convex_function(x_final)), 
                    xytext=(x0, convex_function(x0)),
                    arrowprops=dict(arrowstyle='->', color='red', alpha=0.5))

axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('Convex: All paths lead to same minimum!')
axes[0].grid(True, alpha=0.3)

# Non-convex
axes[1].plot(x_plot, nonconvex_function(x_plot), 'r-', linewidth=2)
results_nonconvex = []
for x0 in starting_points:
    x_final = gd_1d_general(x0, grad_nonconvex, lr=0.01, n_iter=100)
    results_nonconvex.append(x_final)
    axes[1].plot(x0, nonconvex_function(x0), 'ro', markersize=8)
    axes[1].plot(x_final, nonconvex_function(x_final), 'g*', markersize=15)
    axes[1].annotate('', xy=(x_final, nonconvex_function(x_final)), 
                    xytext=(x0, nonconvex_function(x0)),
                    arrowprops=dict(arrowstyle='->', color='red', alpha=0.5))

axes[1].set_xlabel('x')
axes[1].set_ylabel('f(x)')
axes[1].set_title('Non-convex: Different local minima!')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Convex optimization results (from different starting points):")
print(f"  Converged to: {results_convex}")
print(f"  All same? {len(set(np.round(results_convex, 2))) == 1}")

print("\nNon-convex optimization results:")
print(f"  Converged to: {np.round(results_nonconvex, 2)}")
print(f"  Different minima found!")

print("\n" + "="*60)
print("This is why convex optimization is so valuable in ML!")

## Summary

### Key Concepts from Chapter 7:

1. **Gradient Descent**: x_{t+1} = x_t - α∇f(x_t)
2. **Learning Rate**: Controls step size (critical!)
3. **Momentum**: Accelerates convergence
4. **Lagrange Multipliers**: Handle constraints
5. **Convexity**: Guarantees global optimum
6. **KKT Conditions**: Optimality for constrained problems

### Optimization Variants:

| Method | Update Rule | Advantage |
|--------|-------------|------------|
| Batch GD | Full gradient | Stable |
| SGD | Single sample | Fast, online |
| Mini-batch | Batch gradient | Good trade-off |
| Momentum | v = βv + α∇f | Accelerates |
| Adam | Adaptive rates | State-of-the-art |

### ML Applications:

| Concept | ML Application |
|---------|----------------|
| Gradient descent | Training all models |
| SGD | Deep learning |
| Momentum | Faster training |
| Convex optimization | Linear regression, SVM |
| Lagrange multipliers | SVM dual problem |

### Convexity Benefits:

✓ Unique global minimum  
✓ Guaranteed convergence  
✓ Efficient algorithms  
✓ Theoretical guarantees  

### Next Steps:

- **Chapter 9**: Linear Regression (uses gradient descent)
- **Chapter 10**: PCA (eigenvalue optimization)
- **Chapter 12**: SVM (constrained optimization)

---

**Practice**: Implement Adam optimizer and compare with vanilla GD!